In [2]:
import os
os.listdir(".")

['restaurant_ollama_analysis.ipynb',
 'Untitled1.ipynb',
 '.env.txt',
 'restaurant_groq_analysis.ipynb',
 'Untitled.ipynb',
 'Copy_of_2_ollama_and_reviews.ipynb',
 '.env',
 '.ipynb_checkpoints']

In [3]:
from dotenv import dotenv_values
config = dotenv_values(".env")
print(list(config.keys()))

['GROQ_API_KEY']


In [7]:
from dotenv import load_dotenv
import os
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print("key found" if GROQ_API_KEY else "key not found")

key found


In [5]:
%pip install groq

Note: you may need to restart the kernel to use updated packages.


In [8]:
from groq import Groq

In [9]:
client = Groq(api_key=GROQ_API_KEY)

In [8]:
test_review = "The food was delicious, but the service was very slow."
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": test_review}
    ],
    temperature=0
)
print(response.choices[0].message.content)

It sounds like you had a bit of a mixed experience. The food being delicious is definitely a plus, but slow service can be frustrating. Did you end up talking to your server or a manager about the wait, or was it just something you tolerated?


In [10]:
from pydantic import BaseModel
from typing import Optional

class AspectScore(BaseModel):
    score: Optional[float] = None
    quote: Optional[str] = None

class RestaurantReviewAnalysis(BaseModel):
    overall: AspectScore
    food: AspectScore
    service: AspectScore
    price: AspectScore
    environment: AspectScore
    

In [11]:
import json

schema_instructions = RestaurantReviewAnalysis.model_json_schema()

SYSTEM_PROMPT = f"""
You are an expert Restaurant Critic.
Your task is to analyse restaurant reviews and extract sentiment scores for specific aspects.

For each category (Overall, Food, Service, Price, Environment), assign a score between -1.0 and 1.0.
Overall must have a score. Infer the overall sentiment from the full review, even if no explicit overall statement is present.
Never return null for Overall. If the overall sentiment is unclear or mixed, use 0.0.
Environment includes ambience, atmosphere, noise level, cleanliness, decor, comfort, and overall surroundings.
If an aspect receives a score, also provide a short supporting quote from the review whenever possible.
IMPORTANT: Sentiment scores are NOT star ratings.
Never output scores such as 2, 3, 4, or 5.
Every sentiment score must be between -1.0 and +1.0.
A 5-star review does NOT mean a sentiment score of 5.0.
Never output score below -1.0 or above +1.0.
If sentiment is extremely negative, use -1.0.
If sentiment is extremely positive, use +1.0.
Overall score is REQUIRED and can never be null or None.
You MUST always return a numeric overall score between -1.0 and +1.0 for every review.
If the review is mixed or unclear, return 0.0 for overall.

Use:
-1.0 = Strong Negative
0.0 = Neutral / Mixed
+1.0 = Strong Positive
Intermediate values such as -0.7, -0.3, 0.4, or 0.8 are allowed when appropriate.
null = Not Mentioned

Also extract a short quote as evidence.
If the aspect is not mentioned, both the score and quote should be null.

Output ONLY valid JSON that strictly matches this schema:
{json.dumps(schema_instructions, indent=2)}

"""


In [11]:
test_review = "The food was delicious, but the service was very slow."

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": test_review}
    ],
    temperature=0,
    response_format={"type": "json_object"}
)
print(response.choices[0].message.content)

{
  "overall": {
      "score": 0.0,
      "quote": null
   },
   "food": {
      "score": 1.0,
      "quote": "The food was delicious"
   },
   "service": {
      "score": -1.0,
      "quote": "the service was very slow"
   },
   "price": {
      "score": null,
      "quote": null
   },
   "environment": {
      "score": null,
      "quote": null
   }
}


In [12]:
raw_json = response.choices[0].message.content
parsed_sentiment = RestaurantReviewAnalysis.model_validate_json(raw_json)
print(parsed_sentiment)

overall=AspectScore(score=0.0, quote=None) food=AspectScore(score=1.0, quote='The food was delicious') service=AspectScore(score=-1.0, quote='the service was very slow') price=AspectScore(score=None, quote=None) environment=AspectScore(score=None, quote=None)


In [15]:
print("Input tokens.", response.usage.prompt_tokens)
print("Output tokens.", response.usage.completion_tokens)
print("Total tokens.", response.usage.total_tokens)
print("Total time.", response.usage.total_time)


Input tokens. 722
Output tokens. 115
Total tokens. 837
Total time. 0.289052296


In [16]:
from pydantic import BaseModel, Field
from typing import Optional

class GroqUsage(BaseModel):
    input_tokens: Optional[int] = Field(default=None, alias="prompt_tokens")
    output_tokens: Optional[int] = Field(default=None, alias="completion_tokens")
    total_tokens: Optional[int] = None
    total_time: Optional[float] = None

class GroqMetadata(BaseModel):
    model_name: Optional[str] = Field(default=None, alias="model")
    created_at: Optional[int] = Field(default=None, alias="created")
    usage: GroqUsage

In [18]:
metadata_dict = response.model_dump()
parsed_metadata = GroqMetadata.model_validate(metadata_dict)
print(parsed_metadata)

model_name='llama-3.3-70b-versatile' created_at=1786456567 usage=GroqUsage(input_tokens=722, output_tokens=115, total_tokens=837, total_time=0.289052296)


In [19]:
def analyze_review_groq(review_text, review_id, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": review_text}
                ],
                temperature=0,
                response_format={"type": "json_object"}
            )

            return {
                "review_id": review_id,
                "response": response
            }

        except Exception as e:
            print(f"Attempt {attempt + 1} failed for review {review_id}: {e}")

    return None

In [20]:
test_result = analyze_review_groq(
    "The food was delicious, but the service was very slow.",
    review_id="test_1"
)

print(test_result["review_id"])
print(test_result["response"].choices[0].message.content)

test_1
{
  "overall": {
      "score": 0.0,
      "quote": null
   },
   "food": {
      "score": 1.0,
      "quote": "The food was delicious"
   },
   "service": {
      "score": -1.0,
      "quote": "the service was very slow"
   },
   "price": {
      "score": null,
      "quote": null
   },
   "environment": {
      "score": null,
      "quote": null
   }
}


In [21]:
import os
os.listdir("../Data")

['restaurant_ollama_aspects.csv',
 'ollama_metadata.csv',
 'restaurant_reviews_sample.csv',
 'ollama_aspects.csv',
 'restaurant_ollama_metadata.csv']

In [4]:
import pandas as pd
df = pd.read_csv("../Data/restaurant_reviews_sample.csv")
print(df.shape)
df.head()

(200, 5)


,review_id,user_id,business_id,stars,text
0,5dIleZpTxKwjRqBercHV1w,6G7JMGMjwsni6UM-jgn6JA,iSRTaT9WngzB8JJ2YKJUig,5.0,Great meal and service. I didn't realize how l...
1,LETLlxzUzv4eHkD84jea-Q,jOLRUxLpI0NgLsuWHNq9rA,iSRTaT9WngzB8JJ2YKJUig,5.0,Please don't listen to anyone Who gives this p...
2,9sXneDy_Oyd-n-bCUz6TLg,lnl9XpsQ01k-pnxnf-0DAA,iSRTaT9WngzB8JJ2YKJUig,3.0,Seems like a must while in NO. Mostly I enjoye...
3,0tBSNwWA3eMtbBC2PgzQtQ,FJpsfNM620MiQVtGXsznpQ,iSRTaT9WngzB8JJ2YKJUig,1.0,"One of the worst meals I have ever eaten, I wa..."
4,YMiytO01Ocd6rT-0XWfacQ,bK0PskbRUT7Eo6FYOjydlw,iSRTaT9WngzB8JJ2YKJUig,2.0,I had been wanting to try this place for years...


In [23]:
df_zahra = df.iloc[50:100][["review_id", "text"]].copy()

print(df_zahra.shape)
print("First index:", df_zahra.index[0])
print("Last index:", df_zahra.index[-1])
df_zahra.head()

(50, 2)
First index: 50
Last index: 99


,review_id,text
50,jGL1JKYEfco-D9TWhaQe8w,I love the poorboy with ham and steak. It has...
51,QFi1jH4BkmSlPHUqLB7zVg,I ordered the breakfast special & it was ok. M...
52,btJX35dfmsfMYrP2uqE-Cw,Every time I visit New Orleans we drive by Mot...
53,oDUZ2KPhilhoOhe1TLzUBw,Amazing! Everything here is great! I had the h...
54,5BT0-IPs3qwAoQQ50FliGg,Had breakfast here the first Saturday morning ...


In [24]:
test_rows = df_zahra.head(3)

for _, row in test_rows.iterrows():
    result = analyze_review_groq(
        review_text=row["text"],
        review_id=row["review_id"]
    )

    print("Review ID:", result["review_id"])
    print(result["response"].choices[0].message.content)
    print("-" * 50)

Review ID: jGL1JKYEfco-D9TWhaQe8w
{
  "overall": {
      "score": 1.0,
      "quote": "Great lunch. I'll be back when I'm in New Orleans again."
   },
   "food": {
      "score": 1.0,
      "quote": "I love the poorboy with ham and steak. Crawfish etouffe was good."
   },
   "service": {
      "score": null,
      "quote": null
   },
   "price": {
      "score": null,
      "quote": null
   },
   "environment": {
      "score": null,
      "quote": null
   }
}
--------------------------------------------------
Review ID: QFi1jH4BkmSlPHUqLB7zVg
{
  "overall": {
      "score": -0.7,
      "quote": "This place wasn't clean"
   },
   "food": {
      "score": -0.8,
      "quote": "My biscuit was cold & didn't taste good"
   },
   "service": {
      "score": null,
      "quote": null
   },
   "price": {
      "score": 0.5,
      "quote": "Prices are reasonable"
   },
   "environment": {
      "score": -1.0,
      "quote": "This place wasn't clean & the employees that handle the food don't we

In [25]:
test_10 = df_zahra.head(10)

for _, row in test_10.iterrows():
    result = analyze_review_groq(
        review_text=row["text"],
        review_id=row["review_id"]
    )

    print("Review ID:", result["review_id"])
    print(result["response"].choices[0].message.content)
    print("-" * 50)

Review ID: jGL1JKYEfco-D9TWhaQe8w
{
  "overall": {
      "score": 1.0,
      "quote": "Great lunch. I'll be back when I'm in New Orleans again."
   },
   "food": {
      "score": 1.0,
      "quote": "I love the poorboy with ham and steak. Crawfish etouffe was good."
   },
   "service": {
      "score": null,
      "quote": null
   },
   "price": {
      "score": null,
      "quote": null
   },
   "environment": {
      "score": null,
      "quote": null
   }
}
--------------------------------------------------
Review ID: QFi1jH4BkmSlPHUqLB7zVg
{
  "overall": {
      "score": -0.7,
      "quote": "I ordered the breakfast special & it was ok"
   },
   "food": {
      "score": -0.8,
      "quote": "My biscuit was cold & didn't taste good"
   },
   "service": {
      "score": null,
      "quote": null
   },
   "price": {
      "score": 0.5,
      "quote": "Prices are reasonable"
   },
   "environment": {
      "score": -1.0,
      "quote": "This place wasn't clean & the employees that hand

In [12]:
class FinalAspectScore(BaseModel):
    score: Optional[float] = None

class FinalRestaurantReviewAnalysis(BaseModel):
    overall: FinalAspectScore
    food: FinalAspectScore
    service: FinalAspectScore
    price: FinalAspectScore
    environment: FinalAspectScore

In [13]:
final_schema = FinalRestaurantReviewAnalysis.model_json_schema()

FINAL_SYSTEM_PROMPT = f"""
You are an expert Restaurant Critic.
Analyse restaurant reviews and extract sentiment scores for:
Overall, Food, Service, Price, and Environment.

For each aspect, assign a score between -1.0 and +1.0.

Overall MUST always have a numeric score and can never be null.
Infer the overall sentiment from the full review.
If the overall sentiment is mixed or unclear, use 0.0.

Environment includes ambience, atmosphere, noise level, cleanliness,
decor, comfort, and overall surroundings.

Sentiment scores are NOT star ratings.
Never output scores below -1.0 or above +1.0.

Use:
-1.0 = Strong Negative
0.0 = Neutral / Mixed
+1.0 = Strong Positive

Intermediate values such as -0.7, -0.3, 0.4, or 0.8 are allowed.

If an aspect other than Overall is not mentioned, its score should be null.

Output ONLY valid JSON that strictly matches this schema:
{json.dumps(final_schema, indent=2)}
"""

In [14]:
def analyze_review_groq_final(review_text, review_id, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": FINAL_SYSTEM_PROMPT},
                    {"role": "user", "content": review_text}
                ],
                temperature=0,
                response_format={"type": "json_object"}
            )

            return {
                "review_id": review_id,
                "response": response
            }

        except Exception as e:
            print(f"Attempt {attempt + 1} failed for review {review_id}: {e}")
            time.sleep(1)

    return None


 DO NOT RUN AGAIN
results already saved to CSV.

import time
import pandas as pd

sentiment_rows = []
metadata_rows = []

for _, row in df_zahra.iterrows():

    result = analyze_review_groq_final(
        review_text=row["text"],
        review_id=row["review_id"]
    )

    if result is None:
        print("Failed:", row["review_id"])
        continue

    response = result["response"]

    # Parse sentiment JSON
    parsed = FinalRestaurantReviewAnalysis.model_validate_json(
        response.choices[0].message.content
    )

    sentiment_rows.append({
        "review_id": row["review_id"],
        "overall": parsed.overall.score,
        "food": parsed.food.score,
        "service": parsed.service.score,
        "price": parsed.price.score,
        "environment": parsed.environment.score
    })

    # Save metadata
    metadata_rows.append({
        "review_id": row["review_id"],
        "model_name": response.model,
        "created_at": response.created,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
        "total_time": response.usage.total_time
    })

    # Checkpoint after every review
    pd.DataFrame(sentiment_rows).to_csv(
        "../Data/restaurant_groq_sentiment_zahra.csv",
        index=False
    )

    pd.DataFrame(metadata_rows).to_csv(
        "../Data/restaurant_groq_metadata_zahra.csv",
        index=False
    )

    time.sleep(0.2)

print("Done:", len(sentiment_rows))

In [32]:
sent_check = pd.read_csv("../Data/restaurant_groq_sentiment_zahra.csv")
meta_check = pd.read_csv("../Data/restaurant_groq_metadata_zahra.csv")

print("Sentiment rows:", len(sent_check))
print("Metadata rows:", len(meta_check))

Sentiment rows: 50
Metadata rows: 50


In [5]:
missing_reviews = df.iloc[[100, 101]][["review_id", "text"]].copy()
missing_reviews

,review_id,text
100,E1uWye1DXKVYC0onLSpgeA,Adam the guy who eats excessively on the trave...
101,gxHVXz1Ny5eujrSqZiN7Bw,"Staff is wonderful and incredible friendly, es..."


In [15]:
print(analyze_review_groq_final)

<function analyze_review_groq_final at 0x307b7f600>


In [16]:
missing_sentiment_rows = []
missing_metadata_rows = []

for _, row in missing_reviews.iterrows():
    result = analyze_review_groq_final(
        review_text=row["text"],
        review_id=row["review_id"]
    )

    response = result["response"]
    analysis = FinalRestaurantReviewAnalysis.model_validate_json(
        response.choices[0].message.content
    )

    missing_sentiment_rows.append({
        "review_id": row["review_id"],
        "overall": analysis.overall.score,
        "food": analysis.food.score,
        "service": analysis.service.score,
        "price": analysis.price.score,
        "environment": analysis.environment.score
    })

    missing_metadata_rows.append({
        "review_id": row["review_id"],
        "model_name": response.model,
        "created_at": response.created,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
        "total_time": response.usage.total_time
    })

missing_sent = pd.DataFrame(missing_sentiment_rows)

missing_sent

,review_id,overall,food,service,price,environment
0,E1uWye1DXKVYC0onLSpgeA,0.8,1.0,-0.7,None,-0.8
1,gxHVXz1Ny5eujrSqZiN7Bw,0.8,NaN,0.9,None,0.7


In [18]:
missing_sent.to_csv(
    "../Data/groq_aspects_missing2.csv",
    index=False
)    